# 01. Data Exploration

**Objective:** Explore the chest X-ray dataset, understand class distributions, and visualize sample images.

**Dataset:** Kaggle - Chest X-Ray (Pneumonia, Covid-19, Tuberculosis)

---

In [ ]:
import os
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import pandas as pd

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Dataset Structure

In [ ]:
# Set paths
DATA_DIR = Path("../data/raw")

# Check if data directory exists
if not DATA_DIR.exists():
    print(f"❌ Data directory not found: {DATA_DIR}")
    print("\nPlease download the dataset first:")
    print("  ./download_and_train.sh")
    print("OR")
    print("  uv run python download_dataset.py")
else:
    print(f"✓ Data directory: {DATA_DIR.absolute()}")
    print(f"\nContents:")
    for item in DATA_DIR.iterdir():
        if item.is_dir():
            print(f"  📁 {item.name}/")
        else:
            print(f"  📄 {item.name}")

## 2. Class Distribution

In [ ]:
def count_images(data_dir: Path):
    """Count images per class."""
    class_counts = {}
    image_extensions = {'.png', '.jpg', '.jpeg', '.bmp'}
    
    for class_dir in data_dir.iterdir():
        if class_dir.is_dir():
            count = sum(1 for f in class_dir.iterdir() 
                       if f.is_file() and f.suffix.lower() in image_extensions)
            class_counts[class_dir.name] = count
    
    return class_counts

if DATA_DIR.exists():
    class_counts = count_images(DATA_DIR)
    
    # Create DataFrame
    df_counts = pd.DataFrame({
        'Class': list(class_counts.keys()),
        'Count': list(class_counts.values())
    })
    df_counts = df_counts.sort_values('Count', ascending=False)
    
    # Calculate percentages
    total = df_counts['Count'].sum()
    df_counts['Percentage'] = (df_counts['Count'] / total * 100).round(2)
    
    print(f"\n📊 Dataset Statistics")
    print(f"=" * 50)
    print(f"Total Images: {total:,}")
    print(f"Number of Classes: {len(class_counts)}")
    print(f"=" * 50)
    print(f"\n{df_counts.to_string(index=False)}")

In [ ]:
# Visualize class distribution
if DATA_DIR.exists():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Bar chart
    sns.barplot(data=df_counts, x='Class', y='Count', ax=axes[0], palette='viridis')
    axes[0].set_title('Image Count per Class', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Class')
    axes[0].set_ylabel('Count')
    axes[0].tick_params(axis='x', rotation=45)
    
    # Add value labels
    for i, v in enumerate(df_counts['Count']):
        axes[0].text(i, v + 50, str(v), ha='center', fontsize=10)
    
    # Pie chart
    axes[1].pie(df_counts['Count'], labels=df_counts['Class'], autopct='%1.1f%%',
                colors=plt.cm.viridis(np.linspace(0.2, 0.8, len(df_counts))))
    axes[1].set_title('Class Distribution (%)', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('../results/figures/01_class_distribution.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\n✓ Saved: ../results/figures/01_class_distribution.png")

## 3. Sample Images Visualization

In [ ]:
def get_sample_images(data_dir: Path, class_name: str, n_samples: int = 4):
    """Get sample image paths for a class."""
    class_dir = data_dir / class_name
    if not class_dir.exists():
        return []
    
    image_extensions = {'.png', '.jpg', '.jpeg', '.bmp'}
    image_files = [f for f in class_dir.iterdir() 
                   if f.is_file() and f.suffix.lower() in image_extensions]
    
    # Sample randomly
    import random
    if len(image_files) > n_samples:
        image_files = random.sample(image_files, n_samples)
    
    return image_files[:n_samples]

def plot_sample_images(data_dir: Path, n_samples: int = 4):
    """Plot sample images from each class."""
    class_dirs = [d for d in data_dir.iterdir() if d.is_dir()]
    n_classes = len(class_dirs)
    
    fig, axes = plt.subplots(n_classes, n_samples, figsize=(16, 4 * n_classes))
    
    if n_classes == 1:
        axes = axes.reshape(1, -1)
    
    for idx, class_dir in enumerate(sorted(class_dirs)):
        class_name = class_dir.name
        sample_images = get_sample_images(data_dir, class_name, n_samples)
        
        for j, img_path in enumerate(sample_images):
            try:
                img = Image.open(img_path)
                axes[idx, j].imshow(img, cmap='gray')
                axes[idx, j].axis('off')
                
                if j == 0:
                    axes[idx, j].set_ylabel(class_name, fontsize=12, fontweight='bold', rotation=0, labelpad=60)
            except Exception as e:
                axes[idx, j].text(0.5, 0.5, f'Error\n{e}', ha='center', va='center')
                axes[idx, j].axis('off')
    
    plt.suptitle('Sample Chest X-Ray Images by Class', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    return fig

if DATA_DIR.exists():
    fig = plot_sample_images(DATA_DIR, n_samples=4)
    plt.savefig('../results/figures/02_sample_images.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"\n✓ Saved: ../results/figures/02_sample_images.png")

## 4. Image Size Analysis

In [ ]:
def analyze_image_sizes(data_dir: Path):
    """Analyze image dimensions."""
    sizes = []
    image_extensions = {'.png', '.jpg', '.jpeg', '.bmp'}
    
    for class_dir in data_dir.iterdir():
        if class_dir.is_dir():
            for img_path in class_dir.iterdir():
                if img_path.is_file() and img_path.suffix.lower() in image_extensions:
                    try:
                        with Image.open(img_path) as img:
                            sizes.append({
                                'class': class_dir.name,
                                'width': img.width,
                                'height': img.height,
                                'aspect_ratio': img.width / img.height
                            })
                    except:
                        continue
    
    return pd.DataFrame(sizes)

if DATA_DIR.exists():
    df_sizes = analyze_image_sizes(DATA_DIR)
    
    if not df_sizes.empty:
        print(f"📏 Image Size Statistics")
        print(f"=" * 50)
        print(f"\nOverall:")
        print(f"  Width:  {df_sizes['width'].min()} - {df_sizes['width'].max()} (mean: {df_sizes['width'].mean():.1f})")
        print(f"  Height: {df_sizes['height'].min()} - {df_sizes['height'].max()} (mean: {df_sizes['height'].mean():.1f})")
        print(f"  Aspect Ratio: {df_sizes['aspect_ratio'].min():.2f} - {df_sizes['aspect_ratio'].max():.2f} (mean: {df_sizes['aspect_ratio'].mean():.2f})")
        
        print(f"\nPer Class:")
        size_stats = df_sizes.groupby('class').agg({
            'width': ['min', 'max', 'mean'],
            'height': ['min', 'max', 'mean']
        }).round(1)
        print(size_stats)
        
        # Visualize
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        
        # Width distribution
        sns.boxplot(data=df_sizes, x='class', y='width', ax=axes[0])
        axes[0].set_title('Image Width Distribution', fontsize=12, fontweight='bold')
        axes[0].tick_params(axis='x', rotation=45)
        
        # Height distribution
        sns.boxplot(data=df_sizes, x='class', y='height', ax=axes[1])
        axes[1].set_title('Image Height Distribution', fontsize=12, fontweight='bold')
        axes[1].tick_params(axis='x', rotation=45)
        
        plt.tight_layout()
        plt.savefig('../results/figures/03_image_sizes.png', dpi=300, bbox_inches='tight')
        plt.show()
        print(f"\n✓ Saved: ../results/figures/03_image_sizes.png")

## 5. Class Imbalance Analysis

In [ ]:
if DATA_DIR.exists() and 'class_counts' in locals():
    # Calculate class weights
    total = sum(class_counts.values())
    n_classes = len(class_counts)
    
    # Standard class weight calculation
    class_weights = {k: total / (n_classes * v) for k, v in class_counts.items()}
    
    # Normalize weights
    min_weight = min(class_weights.values())
    class_weights_normalized = {k: v / min_weight for k, v in class_weights.items()}
    
    # Display
    df_weights = pd.DataFrame({
        'Class': list(class_weights.keys()),
        'Count': [class_counts[k] for k in class_weights.keys()],
        'Weight': list(class_weights.values()),
        'Normalized Weight': [class_weights_normalized[k] for k in class_weights.keys()]
    })
    
    print(f"⚖️ Class Weights for Imbalanced Dataset")
    print(f"=" * 50)
    print(df_weights.to_string(index=False))
    
    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Raw weights
    sns.barplot(data=df_weights, x='Class', y='Weight', ax=axes[0], palette='Reds_r')
    axes[0].set_title('Class Weights (Raw)', fontsize=12, fontweight='bold')
    axes[0].tick_params(axis='x', rotation=45)
    
    # Normalized weights
    sns.barplot(data=df_weights, x='Class', y='Normalized Weight', ax=axes[1], palette='Blues_r')
    axes[1].set_title('Class Weights (Normalized)', fontsize=12, fontweight='bold')
    axes[1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.savefig('../results/figures/04_class_weights.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"\n✓ Saved: ../results/figures/04_class_weights.png")

## 6. Summary

In [ ]:
if DATA_DIR.exists():
    print("\n" + "=" * 50)
    print("📊 DATA EXPLORATION SUMMARY")
    print("=" * 50)
    print(f"\n✅ Dataset loaded successfully")
    print(f"📁 Total images: {total:,}")
    print(f"🏷️  Number of classes: {n_classes}")
    print(f"⚖️  Class imbalance detected: {'Yes' if max(class_counts.values()) / min(class_counts.values()) > 1.5 else 'No'}")
    print(f"📏 Image sizes: Variable (will be resized to 224x224)")
    print(f"\n📈 Generated figures:")
    print(f"   - 01_class_distribution.png")
    print(f"   - 02_sample_images.png")
    print(f"   - 03_image_sizes.png")
    print(f"   - 04_class_weights.png")
    print(f"\n📁 All figures saved to: ../results/figures/")
    print("=" * 50)